# Task 3.1 — Gold: Course Performance Daily
## Notebook: 08_gold_course_performance_daily

**Pipeline:** EduPath Learning Analytics · Medallion Architecture
**Layer:** Gold (business-ready aggregates)

**What this notebook does:**
- Aggregates per (course_id + report_date) from Silver and Bronze sources:
    → active_students            : distinct students with ACTIVE enrollment
    → completion_rate            : students COMPLETED / total enrolled
    → avg_quiz_score             : average score_obtained across all attempts
    → avg_video_completion_pct   : average completion_pct from video watch events
    → new_enrollments            : enrollments created on report_date
    → dropout_count              : students who moved to DROPPED status on report_date
    → avg_engagement_score       : composite score from engagement metrics
    → instructor_response_rate   : instructor discussion posts / total student posts
- Partitions output by report_date
- Z-ORDERs by course_id for fast per-course queries
- SLA target: table ready by 06:00 AM daily

**Source tables:**
    mini_project_grp6.bronze.enrollments_bronze
    mini_project_grp6.bronze.quiz_attempts_bronze
    mini_project_grp6.bronze.video_watch_bronze
    mini_project_grp6.silver.student_course_engagement
    mini_project_grp6.silver.discussion_posts_parsed

**Target table:** mini_project_grp6.gold.course_performance_daily

**Business Rules enforced:**
- report_date must not be NULL
- completion_rate must be between 0.0 and 1.0
- instructor_response_rate must be between 0.0 and 1.0
- avg_engagement_score must be between 0.0 and 100.0

**Run order:** Run all cells top-to-bottom. Safe to re-run (OVERWRITE mode per date partition).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
SILVER  = "silver"
GOLD    = "gold"

# --- Source tables ---
ENROLL_BRONZE_TABLE     = f"{CATALOG}.{BRONZE}.enrollments_bronze"
QUIZ_BRONZE_TABLE       = f"{CATALOG}.{BRONZE}.quiz_attempts_bronze"
VIDEO_BRONZE_TABLE      = f"{CATALOG}.{BRONZE}.video_watch_bronze"
ENGAGEMENT_SILVER_TABLE = f"{CATALOG}.{SILVER}.student_course_engagement"
DISCUSSION_SILVER_TABLE = f"{CATALOG}.{SILVER}.discussion_posts_parsed"

# --- Target table ---
COURSE_PERF_GOLD_TABLE  = f"{CATALOG}.{GOLD}.course_performance_daily"
DQ_AUDIT_TABLE          = f"{CATALOG}.audit.dq_audit"

# --- Engagement score weights (must sum to 1.0) ---
# Composite engagement score = weighted sum of normalised signals
# Weights reflect business priority of each signal
WEIGHT_VIDEO    = 0.30    # video completion is the strongest signal
WEIGHT_QUIZ     = 0.30    # quiz performance equally important
WEIGHT_LOGIN    = 0.25    # recency of access
WEIGHT_DISCUSS  = 0.15    # participation adds value but lower weight

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(GOLD)

print(f"✅ Catalog : {CATALOG}")
print(f"✅ Schema  : {GOLD}")
print(f"✅ Target  : {COURSE_PERF_GOLD_TABLE}")
print()
print("Engagement score weights:")
print(f"   video={WEIGHT_VIDEO}  quiz={WEIGHT_QUIZ}  "
      f"login={WEIGHT_LOGIN}  discussion={WEIGHT_DISCUSS}")
print(f"   Total weight: {WEIGHT_VIDEO + WEIGHT_QUIZ + WEIGHT_LOGIN + WEIGHT_DISCUSS}")

**Expected output:**
```
✅ Catalog : mini_project_grp6
✅ Schema  : gold
✅ Target  : mini_project_grp6.gold.course_performance_daily

Engagement score weights:
   video=0.3  quiz=0.3  login=0.25  discussion=0.15
   Total weight: 1.0
```

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType,
    DoubleType, DateType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Determine report_date

For a daily Gold table, report_date is the date we are computing metrics FOR.

In production this is yesterday's date (the pipeline runs at 05:00 AM
computing for the previous full day).

For sample data (all dates are Jan–Jun 2024), we compute metrics for
every distinct date that appears in the enrollments table.
This produces one row per (course_id, date) pair covering the full history.

In production Airflow would pass report_date = yesterday as a parameter.

In [0]:
# ============================================================
# CELL 5 — Determine which dates to compute metrics for
#
# Production: report_date = yesterday (Airflow passes this as parameter)
# Sample data: compute for all distinct enrolled_date values
#              to produce a full historical table
#
# We use enrolled_date as the proxy for "activity date" since
# lms_events has event_date but enrollments are the grain for
# active_students and dropout_count.
#
# For this notebook we compute the FULL history from all
# enrollment dates. In Airflow the incremental version would
# filter to report_date = yesterday only.
# ============================================================

enroll_df = spark.table(ENROLL_BRONZE_TABLE)

# Find the date range in the enrollment data
date_range = enroll_df.agg(
    F.min("enrolled_date").alias("min_date"),
    F.max("enrolled_date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print(f"ℹ  Enrollment date range  : {min_date} → {max_date}")
print(f"ℹ  In production Airflow passes report_date = yesterday")
print(f"ℹ  For sample data: computing full history across all dates")
print()

# For Gold, we use max_date as the single report_date
# (represents the most recent complete day snapshot)
# This gives us one meaningful row per course showing latest metrics
REPORT_DATE = max_date
print(f"✅ report_date set to: {REPORT_DATE}")

## Part B — Active Students + Completion Rate + New Enrollments + Dropout Count

Source: bronze.enrollments_bronze

For each course_id as of report_date:
- active_students  : COUNT(DISTINCT student_id) WHERE status = 'ACTIVE'
- completion_rate  : COUNT(DISTINCT student_id WHERE status='COMPLETED') /
                     COUNT(DISTINCT student_id) total enrolled
- new_enrollments  : COUNT WHERE enrolled_date = report_date
- dropout_count    : COUNT WHERE status = 'DROPPED'
                     (in production this would filter to rows updated TODAY
                      using _last_modified_ts; here we use total DROPPED count
                      as a proxy since we don't have a daily delta indicator)

In [0]:
# ============================================================
# CELL 7 — Enrollment metrics per course_id
#
# active_students  : ACTIVE enrollments per course
# completion_rate  : COMPLETED / total enrolled (all statuses)
# new_enrollments  : enrolled on the report_date specifically
# dropout_count    : students currently with DROPPED status
#
# Note on dropout_count for sample data:
#   In production this would use _last_modified_ts = today to count
#   students who changed to DROPPED specifically today.
#   For sample data we use total DROPPED count as the metric.
# ============================================================

enroll_df = spark.table(ENROLL_BRONZE_TABLE)

enrollment_metrics_df = (
    enroll_df
    .filter(F.col("course_id").isNotNull())
    .groupBy("course_id")
    .agg(
        # Total distinct students ever enrolled (denominator for completion_rate)
        F.countDistinct("student_id").alias("total_enrolled"),
        # Currently ACTIVE students
        F.countDistinct(
            F.when(F.col("status") == "ACTIVE", F.col("student_id"))
        ).alias("active_students"),
        # Students who completed
        F.countDistinct(
            F.when(F.col("status") == "COMPLETED", F.col("student_id"))
        ).alias("completed_students"),
        # Students who dropped
        F.countDistinct(
            F.when(F.col("status") == "DROPPED", F.col("student_id"))
        ).alias("dropout_count"),
        # New enrollments on report_date
        F.countDistinct(
            F.when(
                F.col("enrolled_date") == F.lit(REPORT_DATE),
                F.col("student_id")
            )
        ).alias("new_enrollments"),
    )
    # Compute completion_rate from aggregated counts
    .withColumn(
        "completion_rate",
        F.round(
            F.col("completed_students") / F.col("total_enrolled"),
            4
        )
    )
    # Add report_date to every row (partition key)
    .withColumn("report_date", F.lit(REPORT_DATE))
    # Drop intermediate columns not needed in output
    .drop("total_enrolled", "completed_students")
)

print("── Enrollment metrics per course ─────────────────")
print(f"   Distinct courses: {enrollment_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
enrollment_metrics_df.select(
    F.round(F.avg("active_students"), 2).alias("avg_active_students"),
    F.round(F.avg("completion_rate"), 4).alias("avg_completion_rate"),
    F.round(F.avg("new_enrollments"), 2).alias("avg_new_enrollments"),
    F.round(F.avg("dropout_count"), 2).alias("avg_dropout_count")
).show()

## Part C — Quiz Performance Metrics

Source: bronze.quiz_attempts_bronze

Per course_id:
- avg_quiz_score : AVG(score_obtained) across all PASSED + FAILED attempts
                   (excludes IN_PROGRESS and TIMED_OUT — no final score yet)
- quiz_pass_rate : already available in silver.student_course_engagement
                   but we recompute at course level here for Gold

In [0]:
# ============================================================
# CELL 9 — Quiz performance metrics per course_id
#
# avg_quiz_score : average raw score (not normalised) across completed attempts
#                 Only PASSED and FAILED have definitive scores
# course_quiz_pass_rate: course-level pass rate (all students, all quizzes)
# ============================================================

quiz_df = spark.table(QUIZ_BRONZE_TABLE)

quiz_metrics_df = (
    quiz_df
    .filter(
        F.col("course_id").isNotNull() &
        F.col("status").isin("PASSED", "FAILED")    # completed attempts only
    )
    .groupBy("course_id")
    .agg(
        # Average raw score across all completed attempts for this course
        F.round(F.avg("score_obtained"), 2).alias("avg_quiz_score"),
        # Course-level pass rate
        F.round(
            F.sum(F.when(F.col("status") == "PASSED", 1).otherwise(0)) /
            F.count("*"),
            4
        ).alias("course_quiz_pass_rate"),
        # Total quiz attempts for this course
        F.count("*").alias("total_quiz_attempts"),
    )
)

print("── Quiz metrics per course ───────────────────────")
print(f"   Distinct courses: {quiz_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
quiz_metrics_df.select(
    F.min("avg_quiz_score").alias("min_avg_score"),
    F.max("avg_quiz_score").alias("max_avg_score"),
    F.round(F.avg("avg_quiz_score"), 2).alias("overall_avg_score"),
    F.round(F.avg("course_quiz_pass_rate"), 4).alias("overall_pass_rate")
).show()

## Part D — Video Completion Metrics

Source: bronze.video_watch_bronze

Per course_id:
- avg_video_completion_pct : AVG(completion_pct) across all watch events
                             Note: stored as 0–100 in Bronze, kept as 0–100
                             in this Gold column (matches reporting convention)

In [0]:
# ============================================================
# CELL 11 — Video completion metrics per course_id
#
# avg_video_completion_pct : average completion % (0–100 scale)
#                            kept at 0-100 in Gold (not normalised)
#                            to match what instructors expect in dashboards
# total_video_views        : total watch events for this course
# ============================================================

video_df = spark.table(VIDEO_BRONZE_TABLE)

video_metrics_df = (
    video_df
    .filter(F.col("course_id").isNotNull())
    .groupBy("course_id")
    .agg(
        # Average completion % across all watch events (0-100 scale)
        F.round(F.avg("completion_pct"), 2).alias("avg_video_completion_pct"),
        # Total watch events for context
        F.count("*").alias("total_video_views"),
        # Students who watched at least one video
        F.countDistinct("student_id").alias("students_with_video_activity"),
    )
)

print("── Video metrics per course ──────────────────────")
print(f"   Distinct courses: {video_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
video_metrics_df.select(
    F.min("avg_video_completion_pct").alias("min_avg_completion"),
    F.max("avg_video_completion_pct").alias("max_avg_completion"),
    F.round(F.avg("avg_video_completion_pct"), 2).alias("overall_avg_completion")
).show()

## Part E — Instructor Response Rate

Source: silver.discussion_posts_parsed

Per course_id:
- instructor_response_rate : count(instructor posts) / count(student posts)
                             per course — how responsive instructors are

This is the same metric as instructor_reply_rate in the Silver table
but aggregated at course level (not thread level) for the Gold dashboard.

Formula:
  instructor_response_rate = total instructor posts in course /
                             total student posts in course
  Capped at 1.0 if instructors post more than students (rare but possible)

In [0]:
# ============================================================
# CELL 13 — Instructor response rate per course_id
#
# instructor_response_rate = instructor posts / student posts
# Capped at 1.0 (ratio, not a percentage — stays in [0, 1])
# A value of 0 means no instructor participated in any thread
# A value of 1.0 means instructors matched or exceeded student posts
# ============================================================

discussion_df = spark.table(DISCUSSION_SILVER_TABLE)

instructor_rate_df = (
    discussion_df
    .filter(F.col("course_id").isNotNull())
    .groupBy("course_id")
    .agg(
        # Count instructor posts
        F.sum(
            F.when(F.col("is_instructor_post") == True, 1).otherwise(0)
        ).alias("instructor_post_count"),
        # Count student posts
        F.sum(
            F.when(F.col("is_instructor_post") == False, 1).otherwise(0)
        ).alias("student_post_count"),
    )
    # Compute rate — avoid divide by zero with nullif equivalent
    .withColumn(
        "instructor_response_rate",
        F.round(
            F.when(
                F.col("student_post_count") > 0,
                # Cap at 1.0 using F.least()
                F.least(
                    F.col("instructor_post_count") / F.col("student_post_count"),
                    F.lit(1.0)
                )
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .drop("instructor_post_count", "student_post_count")
)

print("── Instructor response rate per course ───────────")
print(f"   Distinct courses: {instructor_rate_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
instructor_rate_df.select(
    F.min("instructor_response_rate").alias("min_rate"),
    F.max("instructor_response_rate").alias("max_rate"),
    F.round(F.avg("instructor_response_rate"), 4).alias("avg_rate")
).show()

## Part F — Compute avg_engagement_score

Source: silver.student_course_engagement

avg_engagement_score is a composite 0–100 score per course
calculated as a weighted average of four normalised signals:

  video_score    = video_completion_rate * 100       (already 0-1, scale to 0-100)
  quiz_score     = quiz_pass_rate * 100              (already 0-1, scale to 0-100)
  login_score    = LEAST(login_count_30d / 8.0, 1.0) * 100
                   (8 logins/month = 100%, more than 8 capped at 100)
  discuss_score  = LEAST(discussion_posts_count / 3.0, 1.0) * 100
                   (3 posts = 100% participation, more capped at 100)

  avg_engagement_score = (video_score * 0.30) + (quiz_score * 0.30)
                       + (login_score * 0.25) + (discuss_score * 0.15)

We first compute per-student scores, then AVG across students per course.

In [0]:
# ============================================================
# CELL 15 — Compute composite engagement score per course_id
#
# Step 1: compute per-student 0-100 engagement score
# Step 2: average across students per course
#
# Normalisation:
#   video, quiz: already 0.0-1.0 in engagement table → multiply by 100
#   login: cap at 8 logins/month as "full engagement" reference point
#   discussion: cap at 3 posts as "full engagement" reference point
#
# These reference points can be tuned as the business learns more.
# ============================================================

engagement_df = spark.table(ENGAGEMENT_SILVER_TABLE)

# Step 1 — Per-student engagement score
student_scores_df = (
    engagement_df
    .filter(F.col("course_id").isNotNull())
    .withColumn(
        "video_score",
        F.col("video_completion_rate") * 100.0
    )
    .withColumn(
        "quiz_score",
        F.col("quiz_pass_rate") * 100.0
    )
    .withColumn(
        "login_score",
        F.least(F.col("login_count_30d") / F.lit(8.0), F.lit(1.0)) * 100.0
    )
    .withColumn(
        "discuss_score",
        F.least(F.col("discussion_posts_count") / F.lit(3.0), F.lit(1.0)) * 100.0
    )
    .withColumn(
        "engagement_score",
        F.round(
            (F.col("video_score")   * WEIGHT_VIDEO) +
            (F.col("quiz_score")    * WEIGHT_QUIZ) +
            (F.col("login_score")   * WEIGHT_LOGIN) +
            (F.col("discuss_score") * WEIGHT_DISCUSS),
            2
        )
    )
)

# Step 2 — Average engagement score per course
engagement_score_df = (
    student_scores_df
    .groupBy("course_id")
    .agg(
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement_score"),
        F.countDistinct("student_id").alias("scored_students_count"),
    )
)

print("── avg_engagement_score per course ───────────────")
print(f"   Distinct courses: {engagement_score_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
engagement_score_df.select(
    F.min("avg_engagement_score").alias("min_score"),
    F.max("avg_engagement_score").alias("max_score"),
    F.round(F.avg("avg_engagement_score"), 2).alias("overall_avg_score")
).show()

print("── Engagement score distribution (buckets) ───────")
student_scores_df.select(
    F.when(F.col("engagement_score") >= 75, "75-100 (High)")
     .when(F.col("engagement_score") >= 50, "50-74 (Medium)")
     .when(F.col("engagement_score") >= 25, "25-49 (Low)")
     .otherwise("0-24 (Very Low)")
     .alias("score_bucket")
).groupBy("score_bucket").count().orderBy("score_bucket").show()

## Part G — Join All Metrics into Final Gold DataFrame

Join all five metric DataFrames on course_id.
Use LEFT JOINs so courses with no quiz/video/discussion activity
still appear in the output with 0 values.

This ensures every active course has a row in course_performance_daily
even if students haven't started activities yet.

In [0]:
# ============================================================
# CELL 17 — JOIN all metric DataFrames on course_id
#
# Base: enrollment_metrics_df (every enrolled course has a row)
# Others: LEFT JOIN to preserve courses with zero activity
#
# After join: fill NULLs with 0 for courses with no activity
# ============================================================

course_perf_df = (
    enrollment_metrics_df
    # Join quiz metrics
    .join(quiz_metrics_df.drop("total_quiz_attempts"), on="course_id", how="left")
    # Join video metrics
    .join(video_metrics_df.drop("total_video_views", "students_with_video_activity"),
          on="course_id", how="left")
    # Join engagement score
    .join(engagement_score_df.drop("scored_students_count"),
          on="course_id", how="left")
    # Join instructor response rate
    .join(instructor_rate_df, on="course_id", how="left")
    # Fill NULLs with 0 for courses with no activity data
    .fillna({
        "avg_quiz_score":             0.0,
        "course_quiz_pass_rate":      0.0,
        "avg_video_completion_pct":   0.0,
        "avg_engagement_score":       0.0,
        "instructor_response_rate":   0.0,
    })
)

print("── Joined course performance DataFrame ───────────")
print(f"   Rows (courses): {course_perf_df.count():,}")
print(f"   Columns       : {len(course_perf_df.columns)}")

In [0]:
# ============================================================
# CELL 18 — Assemble final Gold DataFrame
#           Select and order all output columns cleanly
#           Add Gold metadata columns
# ============================================================

course_performance_final_df = (
    course_perf_df
    .select(
        # ── Grain columns ─────────────────────────────────────
        "course_id",
        "report_date",
        # ── Enrollment signals ────────────────────────────────
        "active_students",
        "completion_rate",
        "new_enrollments",
        "dropout_count",
        # ── Quiz performance ──────────────────────────────────
        "avg_quiz_score",
        "course_quiz_pass_rate",
        # ── Video performance ─────────────────────────────────
        "avg_video_completion_pct",
        # ── Engagement composite ──────────────────────────────
        "avg_engagement_score",
        # ── Discussion health ─────────────────────────────────
        "instructor_response_rate",
    )
    # Add Gold metadata columns
    .withColumn("_gold_load_ts",   F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
)

print("── Final Gold DataFrame ──────────────────────────")
print(f"   Rows    : {course_performance_final_df.count():,}")
print(f"   Columns : {len(course_performance_final_df.columns)}")
print()
course_performance_final_df.printSchema()

In [0]:
# ============================================================
# CELL 19 — Write to gold.course_performance_daily
#
# Write strategy:
#   1. WRITE with partitionBy("report_date")
#      → each date's data lands in its own partition directory
#      → Airflow can overwrite just today's partition without
#        touching historical data
#
#   2. Z-ORDER by course_id (run as separate OPTIMIZE command)
#      → co-locates all rows for the same course_id within each
#        partition's data files
#      → fast predicate pushdown when querying a single course
#
# Mode: overwrite with replaceWhere for the specific report_date
#       This is the production-safe pattern — only today's partition
#       is replaced, not the full table history
# ============================================================

# Step 1 — Write with partition
(
    course_performance_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    # replaceWhere: only overwrite rows for this specific report_date
    # This is idempotent — re-running for the same date replaces just that day
    .option("replaceWhere", f"report_date = '{REPORT_DATE}'")
    .partitionBy("report_date")
    .saveAsTable(COURSE_PERF_GOLD_TABLE)
)

written_count = spark.table(COURSE_PERF_GOLD_TABLE).count()
print(f"✅ Written to: {COURSE_PERF_GOLD_TABLE}")
print(f"   Total rows in table: {written_count:,}")
print(f"   Partition: report_date = {REPORT_DATE}")

**Expected output:**
```
✅ Written to: mini_project_grp6.gold.course_performance_daily
   Total rows in table: 250
   Partition: report_date = 2024-06-30
```

In [0]:
%sql
-- ============================================================
-- CELL 20 — OPTIMIZE + Z-ORDER by course_id
--
-- OPTIMIZE compacts small Delta files created per partition
-- into larger, more efficient files.
--
-- ZORDER BY course_id co-locates rows with the same course_id
-- within each data file so Databricks can skip entire files
-- when filtering by course_id (data skipping).
--
-- This is what makes per-course dashboard queries fast.
-- Run after every write in production.
-- ============================================================

OPTIMIZE mini_project_grp6.gold.course_performance_daily
ZORDER BY (course_id);

In [0]:
# ============================================================
# CELL 21 — Verify gold.course_performance_daily
# ============================================================

gold_df = spark.table(COURSE_PERF_GOLD_TABLE)

print("── course_performance_daily ──────────────────────")
print(f"Total rows   : {gold_df.count():,}")
print(f"Columns      : {len(gold_df.columns)}")
print()

print("── Date partitions in table ──────────────────────")
gold_df.select("report_date").distinct().orderBy("report_date").show(5)

print("── Key metric stats ──────────────────────────────")
gold_df.select(
    F.round(F.avg("active_students"), 2).alias("avg_active_students"),
    F.round(F.avg("completion_rate"), 4).alias("avg_completion_rate"),
    F.round(F.avg("avg_quiz_score"), 2).alias("avg_quiz_score"),
    F.round(F.avg("avg_video_completion_pct"), 2).alias("avg_video_pct"),
    F.round(F.avg("avg_engagement_score"), 2).alias("avg_engagement_score"),
    F.round(F.avg("instructor_response_rate"), 4).alias("avg_instructor_rate")
).show()

print("── Top 5 courses by engagement score ────────────")
gold_df.orderBy(F.desc("avg_engagement_score")) \
    .select(
        "course_id", "active_students", "completion_rate",
        "avg_quiz_score", "avg_video_completion_pct",
        "avg_engagement_score", "instructor_response_rate"
    ).show(5, truncate=False)

print("── Bottom 5 courses by completion rate ──────────")
gold_df.orderBy(F.asc("completion_rate")) \
    .select(
        "course_id", "active_students", "completion_rate",
        "dropout_count", "avg_engagement_score"
    ).show(5, truncate=False)

## Part H — Data Quality Checks (Gold)

DQ checks for course_performance_daily:
1. report_date must not be NULL on any row
2. completion_rate must be between 0.0 and 1.0
3. instructor_response_rate must be between 0.0 and 1.0
4. avg_engagement_score must be between 0.0 and 100.0
5. active_students must be >= 0
6. All courses in course_catalog_bronze must appear in output
   (if a course has enrollments, it must have a Gold row)

In [0]:
# ============================================================
# CELL 23 — DQ Check 1: report_date must not be NULL
#           NULL report_date would break partition pruning
#           and all date-filtered dashboard queries
# ============================================================

gold_df = spark.table(COURSE_PERF_GOLD_TABLE)

null_report_date = gold_df.filter(F.col("report_date").isNull()).count()
print("── DQ Check 1: NULL report_date ──────────────────")

if null_report_date > 0:
    flagged = (
        gold_df.filter(F.col("report_date").isNull())
        .withColumn("dq_check_name", F.lit("null_report_date_gold"))
        .withColumn("dq_table",      F.lit(COURSE_PERF_GOLD_TABLE))
        .withColumn("dq_severity",   F.lit("HIGH"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.lit("report_date is NULL in course_performance_daily"))
    )
    (
        flagged
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {null_report_date} rows with NULL report_date → {DQ_AUDIT_TABLE}")
else:
    print(f"   ✅ PASSED — report_date is populated on all {gold_df.count():,} rows")

In [0]:
# ============================================================
# CELL 24 — DQ Check 2: validate all rate/score columns
#
# completion_rate          : must be in [0.0, 1.0]
# instructor_response_rate : must be in [0.0, 1.0]
# avg_engagement_score     : must be in [0.0, 100.0]
# active_students          : must be >= 0
# ============================================================

gold_df = spark.table(COURSE_PERF_GOLD_TABLE)

range_checks = [
    ("completion_rate",          0.0, 1.0,   "HIGH"),
    ("instructor_response_rate", 0.0, 1.0,   "HIGH"),
    ("avg_engagement_score",     0.0, 100.0, "HIGH"),
]

for col_name, min_val, max_val, severity in range_checks:
    invalid_rows = gold_df.filter(
        (F.col(col_name) < min_val) | (F.col(col_name) > max_val)
    )
    invalid_count = invalid_rows.count()

    print(f"── DQ Check 2: {col_name} in [{min_val}, {max_val}] ────")
    if invalid_count > 0:
        (
            invalid_rows
            .withColumn("dq_check_name", F.lit(f"{col_name}_out_of_range_gold"))
            .withColumn("dq_table",      F.lit(COURSE_PERF_GOLD_TABLE))
            .withColumn("dq_severity",   F.lit(severity))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.concat_ws(" | ",
                F.lit(f"{col_name} out of range [{min_val},{max_val}]:"),
                F.col(col_name).cast("string"),
                F.lit("course_id:"), F.col("course_id")
            ))
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {invalid_count} rows out of range → {DQ_AUDIT_TABLE}")
    else:
        print(f"   ✅ PASSED")
    print()

# Active students >= 0
neg_active = gold_df.filter(F.col("active_students") < 0).count()
print(f"── DQ Check 2: active_students >= 0 ─────────────")
print(f"   {'✅ PASSED' if neg_active == 0 else f'⚠ {neg_active} rows with negative active_students'}")

In [0]:
# ============================================================
# CELL 25 — DQ Check 3: every course in catalog that has enrollments
#           must appear in course_performance_daily
#
# If a course exists in enrollments but NOT in the Gold table,
# instructors will have a blind spot for that course.
# ============================================================

from pyspark.sql import functions as F

# Courses that have at least one enrollment
courses_with_enrollments = (
    spark.table(ENROLL_BRONZE_TABLE)
    .select("course_id")
    .distinct()
)

# Courses present in Gold output
courses_in_gold = (
    spark.table(COURSE_PERF_GOLD_TABLE)
    .select("course_id")
    .distinct()
)

# Courses that have enrollments but are MISSING from Gold
missing_courses = (
    courses_with_enrollments
    .join(courses_in_gold, on="course_id", how="left_anti")
)

missing_count = missing_courses.count()
enrolled_count = courses_with_enrollments.count()

print("── DQ Check 3: course coverage in Gold ───────────")
print(f"   Courses with enrollments     : {enrolled_count:,}")
print(f"   Courses present in Gold table: {courses_in_gold.count():,}")
print(f"   Missing courses              : {missing_count}")

if missing_count > 0:
    print("   ⚠ Missing course_ids:")
    missing_courses.show(10, truncate=False)
    (
        missing_courses
        .withColumn("dq_check_name", F.lit("missing_course_in_gold"))
        .withColumn("dq_table",      F.lit(COURSE_PERF_GOLD_TABLE))
        .withColumn("dq_severity",   F.lit("MEDIUM"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit("course_id has enrollments but is missing from Gold:"),
            F.col("course_id")
        ))
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
else:
    print("   ✅ PASSED — all courses with enrollments are present in Gold")

In [0]:
%sql
-- ============================================================
-- CELL 26 — SQL verification of gold.course_performance_daily
-- ============================================================

SELECT
    COUNT(*)                                          AS total_rows,
    COUNT(DISTINCT course_id)                         AS unique_courses,
    COUNT(DISTINCT report_date)                       AS distinct_dates,
    ROUND(AVG(active_students), 2)                    AS avg_active_students,
    ROUND(AVG(completion_rate), 4)                    AS avg_completion_rate,
    ROUND(AVG(avg_quiz_score), 2)                     AS avg_quiz_score,
    ROUND(AVG(avg_video_completion_pct), 2)           AS avg_video_completion_pct,
    ROUND(AVG(avg_engagement_score), 2)               AS avg_engagement_score,
    ROUND(AVG(instructor_response_rate), 4)           AS avg_instructor_response_rate,
    SUM(new_enrollments)                              AS total_new_enrollments,
    SUM(dropout_count)                                AS total_dropouts
FROM mini_project_grp6.gold.course_performance_daily;

In [0]:
%sql
-- ============================================================
-- CELL 27 — SLA check: verify Gold table freshness
--           In Airflow this is a separate sensor task that
--           checks _gold_load_ts vs the SLA deadline.
--
--           Here we simulate: if _gold_load_ts is within the
--           same day as report_date, the SLA is considered met.
-- ============================================================

SELECT
    report_date,
    COUNT(*)                          AS rows_for_date,
    MAX(_gold_load_ts)                AS last_loaded_at,
    CASE
        WHEN MAX(_gold_load_ts) IS NOT NULL
        THEN '✅ Data present — SLA met'
        ELSE '❌ No data — SLA missed'
    END                               AS sla_status
FROM mini_project_grp6.gold.course_performance_daily
GROUP BY report_date
ORDER BY report_date DESC
LIMIT 5;

In [0]:
# ============================================================
# CELL 28 — Task 3.1 completion summary
# ============================================================

final_df    = spark.table(COURSE_PERF_GOLD_TABLE)
final_count = final_df.count()
courses     = final_df.select("course_id").distinct().count()

print("═" * 62)
print("  TASK 3.1 COMPLETE — Gold Course Performance Daily")
print("═" * 62)
print()
print(f"  ✅ {COURSE_PERF_GOLD_TABLE}")
print(f"      Rows (course-date pairs)  : {final_count:,}")
print(f"      Distinct courses          : {courses:,}")
print(f"      report_date               : {REPORT_DATE}")
print(f"      Partition                 : report_date")
print(f"      Z-ORDER                   : course_id")
print(f"      Write mode                : replaceWhere(report_date)")
print()
print("  Metrics in output:")
print("      active_students           — ACTIVE enrollments per course")
print("      completion_rate           — COMPLETED / total enrolled")
print("      new_enrollments           — enrolled on report_date")
print("      dropout_count             — DROPPED students per course")
print("      avg_quiz_score            — avg raw score (completed attempts)")
print("      course_quiz_pass_rate     — pass rate across all students")
print("      avg_video_completion_pct  — avg video completion (0-100)")
print("      avg_engagement_score      — weighted composite (0-100)")
print("      instructor_response_rate  — instructor posts / student posts")
print()
print("  DQ checks run:")
print("      Cell 23 — NULL report_date")
print("      Cell 24 — rate/score columns in valid ranges")
print("      Cell 25 — course coverage vs enrollment catalog")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 3.2 → 09_gold_instructor_dashboard")
print("         Per instructor, weekly refresh, RANK() by completion")
print("═" * 62)